# 03 — Train single CatBoost on the 1-min dataset

Single model (no ensemble), block-bootstrap CIs for honest uncertainty on
overlapping 1-min labels, training-period score quantiles persisted to
`predictions_metadata_1min.json` for downstream consumers (so strategy
scripts pick operating thresholds from train without peeking at val+test).

Configuration:

- **CatBoost fixed params** come from `src.utils.CB_FIXED_PARAMS` (legacy):
  Ordered boosting (`has_time=True`), MVS bootstrap, Langevin SGLB, Newton
  leaf estimation, GreedyLogSum quantization with `border_count=128`.
- **Best hyperparameters** override the fixed params with the legacy
  research model: `learning_rate=0.01`, `depth=6`, `l2_leaf_reg=5.0`,
  `rsm=1.0`, `subsample=1.0`, `mvs_reg=3.0`, `diffusion_temperature=10000`,
  `random_strength=1.5`.
- **Sample weights = barrier_distance × label_uniqueness.** The
  asymmetric barrier-distance weight is read from the `weight` column
  produced by `02_build_features` (near-miss negatives exponentially
  upweighted, range ~[1, 5]). It is multiplied row-wise by the Lopez de
  Prado average-uniqueness `u_i = mean(1/c_l, l in I_i)` computed
  per-split via `src.analytics.sampling.compute_uniqueness_weights`
  (`normalize=False`, `bar_stride=1`). Interior 1-min rows get `u ≈ 1/M`,
  so the gradient sees roughly `n/M` independent labels — the correct
  effective N for overlapping triple-barrier targets. Both weighting
  legs are passed to the CatBoost Pool for train and val so the
  val-loss curve driving `best_iter` is computed under the same
  weighting the model is fit against.
- **Early stopping** stays ON (`early_stopping_rounds=100`) as a
  *diagnostic*. It fires when the val signal is exhausted, which usually
  signals a problem **upstream** (label/weight construction, train/val
  split alignment, embargo, feature parity) rather than a hyperparameter
  problem. The cell logs loudly when it fires but does not raise. With
  uniqueness weighting active, `best_iter` may land much lower than the
  pre-uniqueness ~700 — that is expected, not a regression.

HPO is intentionally skipped — at ~500k rows × ~1.2k features each Optuna
trial costs many minutes. The hyperparameters above were selected via the
legacy boundary-cadence research; HPO can be revisited if the headline
numbers warrant the compute.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

# Resolve repo root (works whether Jupyter is launched from repo root or notebooks/)
ROOT = Path.cwd()
if not (ROOT / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
    if (ROOT.parent / 'docs' / 'MINIMAL_PROJECT_SPEC_v2.md').exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError('Could not locate repo root.')
sys.path.insert(0, str(ROOT))

from src import utils
from src.analytics.bootstrap import bootstrap_metric
from src.analytics.sampling import compute_uniqueness_weights
from src.analytics.uncertainty import (
    predictive_uncertainty,
    virtual_ensemble_predictions,
)
from src.features.config import M, PHI
from src.utils import get_git_sha

warnings.filterwarnings('ignore')

# ----- Output paths --------------------------------------------------------
DATASET_DIR = ROOT / 'data' / 'model_dataset'
DATASET_PATH = DATASET_DIR / 'dataset_1min.parquet'
FEATURE_LIST_PATH = DATASET_DIR / 'feature_list_1min.json'
MODEL_PATH = DATASET_DIR / 'catboost_model_1min.cbm'
PREDICTIONS_PATH = DATASET_DIR / 'research_predictions_1min.parquet'
METRICS_PATH = DATASET_DIR / 'analytics' / 'research_metrics_1min.json'
META_PATH = DATASET_DIR / 'dataset_metadata_1min.json'
PREDICTIONS_META_PATH = DATASET_DIR / 'predictions_metadata_1min.json'
# Virtual-ensemble uncertainty artifacts (read by 04_offline_evaluation
# and 05_strategy_calibration). K=10 snapshots follow the legacy convention.
TRAIN_UNC_PATH = DATASET_DIR / 'analytics' / 'train_scores_unc_1min.parquet'
VAL_TEST_UNC_PATH = DATASET_DIR / 'analytics' / 'val_test_ve_unc_1min.parquet'
VE_K = 10

# Quantile levels persisted in PREDICTIONS_META_PATH for downstream
# threshold selection.
TRAIN_P_QUANTILE_LEVELS: tuple[float, ...] = (
    0.05, 0.10, 0.50, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95,
)

# ----- Training config -----------------------------------------------------
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
EMBARGO_K = utils.recommended_embargo_for_cadence('1min', base_embargo=60, M=M)

# Restore the legacy training stack:
#
# - CB_FIXED_PARAMS provides the regularization-heavy backbone:
#   `boosting_type='Ordered'` + `has_time=True` (time-aware, no future
#   leakage in the boosting rounds), `bootstrap_type='MVS'` (importance
#   sampling on gradients), `langevin=True` (SGLB — stochastic gradient
#   Langevin boost, the prerequisite for a meaningful virtual ensemble),
#   `leaf_estimation_method='Newton'` with 10 inner iterations,
#   `border_count=128`. Includes `early_stopping_rounds=100`.
#
# - Best hyperparameters (from the legacy boundary-cadence research):
#   `learning_rate=0.01`, `depth=6`, `l2_leaf_reg=5.0`, `rsm=1.0`,
#   `subsample=1.0`, `mvs_reg=3.0`, `diffusion_temperature=10000`,
#   `random_strength=1.5`.
#
# Early stopping is intentionally left ON. We treat it as a diagnostic:
# if it fires very early, the data pipeline upstream is the suspect (label
# construction, weight construction, train/val split alignment, embargo,
# feature parity), not these hyperparameters. The fit cell logs the
# diagnostic loudly but does not raise.
LEGACY_BEST_PARAMS = {
    'learning_rate': 0.01,
    'l2_leaf_reg': 0.1,
    'depth': 6,
    'rsm': 1.0,
    'subsample': 1.0,
    'mvs_reg': 3.0,
    'diffusion_temperature': 10000,
    'random_strength': 0.0,
}

CB_PARAMS = {
    **utils.CB_FIXED_PARAMS,
    **LEGACY_BEST_PARAMS,
    'iterations': 2000,
    'verbose': 20,
}
# Sanity: the legacy stack requires has_time=True (time-aware boosting).
if not CB_PARAMS.get('has_time', False):
    raise RuntimeError(
        'CB_FIXED_PARAMS lost has_time=True; refuse to train without '
        'time-awareness on overlapping 1-min labels.'
    )

print(f'ROOT: {ROOT}')
print(f'Embargo at 1-min cadence: {EMBARGO_K} rows  ({EMBARGO_K / M:.1f} boundaries = {EMBARGO_K / 60:.1f} hours)')

ROOT: C:\Users\vitil\OneDrive\Desktop\barrier_classifier
Embargo at 1-min cadence: 1200 rows  (60.0 boundaries = 20.0 hours)


## 1. Load dataset + feature list

In [2]:
print(f'Loading {DATASET_PATH} ...')
t0 = time.perf_counter()
df = pd.read_parquet(DATASET_PATH)
feature_list = utils.load_json(FEATURE_LIST_PATH)
print(f'  loaded in {time.perf_counter()-t0:.1f}s: {len(df):,} rows x {len(df.columns)} cols')
print(f'  features in feature_list: {len(feature_list):,}')
print(f'  base rate y = {float(df["y"].mean()):.4f}')

# Sanity: every feature in feature_list exists in the dataset.
missing = [c for c in feature_list if c not in df.columns]
if missing:
    raise ValueError(f'feature_list missing from dataset: {missing[:5]}...')

Loading C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\dataset_1min.parquet ...


  loaded in 2.2s: 505,421 rows x 1621 cols
  features in feature_list: 1,511
  base rate y = 0.2100


## 2. Chronological split with cadence-aware embargo

In [3]:
df = df.sort_values('k').reset_index(drop=True)
train_df, val_df, test_df = utils.chronological_split_with_embargo(
    df, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, embargo_k=EMBARGO_K
)

print('Splits:')
print(f"  train: n={len(train_df):,}  k=[{train_df['k'].min()}, {train_df['k'].max()}]  base_rate={float(train_df['y'].mean()):.4f}")
print(f"  val  : n={len(val_df):,}  k=[{val_df['k'].min()}, {val_df['k'].max()}]  base_rate={float(val_df['y'].mean()):.4f}")
print(f"  test : n={len(test_df):,}  k=[{test_df['k'].min()}, {test_df['k'].max()}]  base_rate={float(test_df['y'].mean()):.4f}")

# Runtime check (not a Python assert): under `python -O` asserts are stripped,
# but the embargo invariant is a hard correctness gate.
gap_train_val = int(val_df['k'].min() - train_df['k'].max())
gap_val_test = int(test_df['k'].min() - val_df['k'].max())
if not (gap_train_val >= M and gap_val_test >= M):
    raise RuntimeError(
        f'Insufficient embargo gap between splits: '
        f'train|val={gap_train_val}, val|test={gap_val_test} (need >= M={M}). '
        'Re-check chronological_split_with_embargo / EMBARGO_K.'
    )
print(f'  embargo gaps: train|val={gap_train_val} val|test={gap_val_test} (>= M={M} required)')

Splits:
  train: n=353,794  k=[20159, 373952]  base_rate=0.2049
  val  : n=74,613  k=[375153, 449765]  base_rate=0.1900
  test : n=74,614  k=[450966, 525579]  base_rate=0.2521
  embargo gaps: train|val=1201 val|test=1201 (>= M=20 required)


## 3. Dataset-integrity checks + sample weights (barrier-distance × uniqueness)

Before training: explicit hard-fail checks that the dataset that lands
in CatBoost is bit-identical to what `02_build_features` produced —
no silent NaN substitution, no row dropping beyond the documented
embargo, no dtype surprises, no clipped label values.

Then: build the per-row training weight as the product of two effects:

1. **Asymmetric barrier-distance weight** — read from the `weight`
   column produced by `02_build_features`. Near-miss negatives carry
   `w_dist ∈ (1, 5]`; positives keep `w_dist = 1`.
2. **Label uniqueness** — `u_i = mean(1/c_l, l ∈ I_i)` where `I_i =
   [i+1, i+M)` is row `i`'s label window and `c_l` is the count of
   overlapping label windows that cover bar `l`. Computed per-split via
   `compute_uniqueness_weights(n_rows=len(split), M=M, bar_stride=1,
   normalize=False)`. Interior 1-min rows get `u ≈ 1/M = 0.05`;
   boundary rows approach 1. This shrinks effective N from raw row
   count to roughly `n/M`, which is the correct number of independent
   labels on overlapping triple-barrier targets.

Per-split uniqueness (not full-dataset-then-slice) is correct because
the embargo gap (`EMBARGO_K = 60·M = 1200` rows) vastly exceeds `M = 20`,
so no inter-split label overlap exists — each split is a contiguous
1-min row range and within-split uniqueness is exactly right.

In [4]:
# ===== DATASET-INTEGRITY GATE =============================================
# Hard-fails if any of the following happens between 02_build_features and
# the CatBoost Pool: silent NaN imputation, silent inf clipping, silent
# row dropping beyond embargo, label dtype coercion that loses {0,1},
# dataset row count drift versus the persisted metadata.
# ==========================================================================

# 1) Row-count conservation: train + val + test + embargo_dropped == len(df)
n_total = len(df)
n_in_splits = len(train_df) + len(val_df) + len(test_df)
embargo_dropped = n_total - n_in_splits
expected_dropped = 2 * EMBARGO_K
if embargo_dropped != expected_dropped:
    raise RuntimeError(
        f'Row-count conservation failed: total={n_total:,}, '
        f'in splits={n_in_splits:,}, embargo dropped={embargo_dropped} '
        f'(expected exactly 2*EMBARGO_K = {expected_dropped}). '
        'Something else is filtering rows between split and training.'
    )
print(f'[integrity] row conservation OK: '
      f'{n_total:,} = {len(train_df):,} (train) + {len(val_df):,} (val) '
      f'+ {len(test_df):,} (test) + {embargo_dropped} (embargo)')

# 2) Dataset row count matches what 02_build_features wrote in metadata.
with open(META_PATH) as _f:
    _dataset_meta = json.load(_f)
if int(_dataset_meta['n_rows']) != n_total:
    raise RuntimeError(
        f'Dataset row count drifted: parquet has {n_total:,} rows but '
        f'dataset_metadata_1min.json reports {_dataset_meta["n_rows"]:,}. '
        'Re-run 02_build_features.ipynb to refresh both consistently.'
    )
print(f'[integrity] parquet rows ({n_total:,}) match metadata ({_dataset_meta["n_rows"]:,})')

# 3) Labels: every train/val/test row has y in exactly {0, 1}. NaN-in-y
# would be a real deformation — astype(int) on NaN would silently produce
# the platform's smallest int (e.g. -9223372036854775808), which CatBoost
# would happily train on.
for _name, _sub in [('train', train_df), ('val', val_df), ('test', test_df)]:
    _y = _sub['y']
    _n_nan = int(_y.isna().sum())
    if _n_nan:
        raise RuntimeError(f'{_name}.y has {_n_nan} NaN; labels deformed.')
    _unique = set(np.unique(_y.astype(int).to_numpy()).tolist())
    if not _unique.issubset({0, 1}):
        raise RuntimeError(
            f'{_name}.y has values outside {{0, 1}}: {sorted(_unique)}; '
            'labels deformed.'
        )
print(f'[integrity] labels OK: all train/val/test y are in {{0, 1}}, no NaN')

# 4) Features: no NaN, no inf in the numeric matrix that CatBoost will see.
# The pipeline's imputation step (02) asserts zero null AND zero NaN AND
# zero inf post-impute. If anything slipped through into the parquet,
# CatBoost would receive a deformed dataset.
for _name, _sub in [('train', train_df), ('val', val_df), ('test', test_df)]:
    _X = _sub[feature_list].to_numpy(dtype=float)
    _n_nan = int(np.isnan(_X).sum())
    _n_inf = int(np.isinf(_X).sum())
    if _n_nan or _n_inf:
        # Localize first bad cells for diagnosis.
        if _n_nan:
            _r, _c = np.where(np.isnan(_X))
            _samples = [(int(_r[i]), feature_list[int(_c[i])]) for i in range(min(5, len(_r)))]
        else:
            _samples = []
        raise RuntimeError(
            f'{_name} has {_n_nan} NaN and {_n_inf} inf in features. '
            f'First bad (row, col): {_samples}. Dataset is deformed; '
            'fix 02_build_features impute step before retraining.'
        )
print(f'[integrity] features OK: zero NaN, zero inf across all splits')

# 5) Feature-list ↔ dataset alignment: every column in feature_list is
# present in the dataframe and is a numeric dtype CatBoost can consume.
_missing = [c for c in feature_list if c not in df.columns]
if _missing:
    raise RuntimeError(f'feature_list references columns not in dataset: {_missing[:5]}...')
_dtypes = df[feature_list].dtypes
_nonnum = [(c, str(_dtypes[c])) for c in feature_list if not np.issubdtype(_dtypes[c], np.number)]
if _nonnum:
    raise RuntimeError(f'feature_list has non-numeric columns: {_nonnum[:5]}')
if 'weight' in feature_list:
    raise RuntimeError(
        "Asymmetric `weight` column leaked into feature_list; fix 02_build_features."
    )
print(f'[integrity] feature_list OK: {len(feature_list):,} columns, all numeric, all present')

# ===== ASYMMETRIC LOSS WEIGHTING * LABEL UNIQUENESS =======================
# Two weighting effects are combined into the per-row training weight:
#
# (1) Barrier-distance asymmetric weight: materialized in 02_build_features
#     and read from the `weight` column. Near-miss negatives are
#     exponentially upweighted; positives keep w=1. Range ~ [1, 5].
#
# (2) Lopez de Prado average-uniqueness u_i = mean(1/c_l, l in I_i), where
#     I_i = [i+1, i+M) is row i's label window and c_l is the number of
#     overlapping label windows that cover bar l. Interior 1-min rows on
#     a contiguous slice get u ~ 1/M (here 1/20 = 0.05); boundary rows
#     approach 1. This shrinks effective N from the raw row count to
#     roughly n/M, which is the *correct* number of independent labels
#     for gradient boosting to "see" on overlapping triple-barrier targets.
#
# Per-split computation is correct: the embargo gap (EMBARGO_K = 60*M = 1200)
# vastly exceeds M=20, so no inter-split label overlap exists, and each
# split's contiguous 1-min row range is the right window for uniqueness.
# normalize=False is the documented default for this combination (see
# src/analytics/sampling.py); normalize=True would cancel the M-fold
# downweighting and reintroduce the redundant-label problem.
# ==========================================================================
if 'weight' not in df.columns:
    raise RuntimeError(
        'Dataset is missing the `weight` column. Re-run 02_build_features.ipynb '
        'to materialize the asymmetric loss weights.'
    )
w_dist_train = train_df['weight'].to_numpy(dtype=float)
w_dist_val = val_df['weight'].to_numpy(dtype=float)

u_train = compute_uniqueness_weights(
    n_rows=len(train_df), M=int(M), bar_stride=1, normalize=False,
)
u_val = compute_uniqueness_weights(
    n_rows=len(val_df), M=int(M), bar_stride=1, normalize=False,
)

train_weights = w_dist_train * u_train
val_weights = w_dist_val * u_val

# Hard-fails: weights must be positive and finite.
for _name, _w in [
    ('w_dist_train', w_dist_train), ('w_dist_val', w_dist_val),
    ('u_train', u_train), ('u_val', u_val),
    ('train_weights', train_weights), ('val_weights', val_weights),
]:
    if not np.isfinite(_w).all():
        raise RuntimeError(f'{_name} has non-finite values; weights deformed.')
    if (_w <= 0).any():
        raise RuntimeError(f'{_name} has non-positive values; weights deformed.')

_w_meta = _dataset_meta.get('weighting', {})
print(f'[weights] combined = barrier_distance * lopez_de_prado_avg_uniqueness '
      f'(barrier scheme={_w_meta.get("scheme", "unknown")}, normalize=False):')
print(f'  w_dist train : range [{w_dist_train.min():.4f}, {w_dist_train.max():.4f}]  '
      f'mean={w_dist_train.mean():.4f}')
print(f'  w_dist val   : range [{w_dist_val.min():.4f}, {w_dist_val.max():.4f}]  '
      f'mean={w_dist_val.mean():.4f}')
print(f'  u_train      : range [{u_train.min():.4f}, {u_train.max():.4f}]  '
      f'mean={u_train.mean():.4f}  (interior approx 1/M = {1.0/int(M):.4f})')
print(f'  u_val        : range [{u_val.min():.4f}, {u_val.max():.4f}]  '
      f'mean={u_val.mean():.4f}')
print(f'  combined train: range [{train_weights.min():.4f}, {train_weights.max():.4f}]  '
      f'mean={train_weights.mean():.4f}  sum={train_weights.sum():.1f}')
print(f'  combined val  : range [{val_weights.min():.4f}, {val_weights.max():.4f}]  '
      f'mean={val_weights.mean():.4f}  sum={val_weights.sum():.1f}')
print(f'  effective n (uniqueness only, train) = sum(u_train) = {u_train.sum():.1f}  '
      f'(raw n train = {len(train_df):,}, M-deflated estimate = {len(train_df)/int(M):.1f})')
print(f'  barrier-scheme effective_n (full dataset, from 02 metadata): '
      f'{_w_meta.get("effective_n", float("nan")):.1f}  (raw n full = {n_total:,})')

[integrity] row conservation OK: 505,421 = 353,794 (train) + 74,613 (val) + 74,614 (test) + 2400 (embargo)
[integrity] parquet rows (505,421) match metadata (505,421)
[integrity] labels OK: all train/val/test y are in {0, 1}, no NaN


[integrity] features OK: zero NaN, zero inf across all splits


[integrity] feature_list OK: 1,511 columns, all numeric, all present
[weights] combined = barrier_distance * lopez_de_prado_avg_uniqueness (barrier scheme=barrier_distance_asymmetric, normalize=False):
  w_dist train : range [1.0000, 5.0000]  mean=2.6542
  w_dist val   : range [1.0000, 5.0000]  mean=2.7075
  u_train      : range [0.0500, 0.1799]  mean=0.0500  (interior approx 1/M = 0.0500)
  u_val        : range [0.0500, 0.1799]  mean=0.0500
  combined train: range [0.0500, 0.5244]  mean=0.1327  sum=46953.9
  combined val  : range [0.0500, 0.5052]  mean=0.1354  sum=10103.1
  effective n (uniqueness only, train) = sum(u_train) = 17690.6  (raw n train = 353,794, M-deflated estimate = 17689.7)
  barrier-scheme effective_n (full dataset, from 02 metadata): 397026.9  (raw n full = 505,421)


## 4. Train CatBoost (single model)

In [5]:
print('Training CatBoost with params:')
for k, v in CB_PARAMS.items():
    print(f'  {k}: {v}')

# Pass `weight=` for both train and val pools so the val-loss curve that
# drives best_iter is computed on the same weighting the model is fit
# against.
train_pool = Pool(
    data=train_df[feature_list].to_numpy(),
    label=train_df['y'].astype(int).to_numpy(),
    timestamp=train_df['k'].to_numpy(dtype=np.uint32),
    weight=train_weights,
    feature_names=list(feature_list),
)
val_pool = Pool(
    data=val_df[feature_list].to_numpy(),
    label=val_df['y'].astype(int).to_numpy(),
    timestamp=val_df['k'].to_numpy(dtype=np.uint32),
    weight=val_weights,
    feature_names=list(feature_list),
)

# Hard-fail sanity: Pool row counts must match the DataFrames exactly.
if train_pool.num_row() != len(train_df):
    raise RuntimeError(
        f'CatBoost Pool dropped train rows: '
        f'pool={train_pool.num_row():,} vs df={len(train_df):,}. '
        'Refuse to train on a deformed pool.'
    )
if val_pool.num_row() != len(val_df):
    raise RuntimeError(
        f'CatBoost Pool dropped val rows: '
        f'pool={val_pool.num_row():,} vs df={len(val_df):,}. '
        'Refuse to train on a deformed pool.'
    )
print(f'[integrity] Pool row counts match: train={train_pool.num_row():,}, val={val_pool.num_row():,}')

model = CatBoostClassifier(**CB_PARAMS)
t0 = time.perf_counter()
print(f"Fit start: {time.strftime('%H:%M:%S')}")
model.fit(train_pool, eval_set=val_pool)
fit_dt = time.perf_counter() - t0

# With use_best_model=True (legacy CB_FIXED_PARAMS), CatBoost truncates
# the saved model to best_iter+1 trees. tree_count_ then reflects that
# truncation; best_iteration_ stays as the val-best iteration.
best_iter = int(model.get_best_iteration())
tree_count = int(model.tree_count_)
n_iters_planned = int(CB_PARAMS['iterations'])
es_rounds = int(CB_PARAMS.get('early_stopping_rounds', 0))

# Early stopping stays ON intentionally. It is a diagnostic, not a
# configuration we want to defeat. If `best_iter` lands suspiciously
# early (rule of thumb: << 30% of planned iterations), the data
# pipeline upstream is the suspect — check label/weight construction,
# train/val split alignment, embargo, feature parity across splits —
# *before* tweaking hyperparameters. We log loudly but do not raise so
# the run completes and the artifacts reflect the actual model.
es_fired = es_rounds > 0 and best_iter < (n_iters_planned - 1)
suspiciously_early = best_iter < int(0.30 * n_iters_planned)
if es_fired:
    print(
        f'[diagnostic] Early stopping fired at iter {best_iter} '
        f'(planned {n_iters_planned}, early_stopping_rounds={es_rounds}).'
    )
    if suspiciously_early:
        print(
            '[diagnostic] best_iter is < 30% of planned iterations — investigate '
            'data prep upstream: label/weight construction, train/val split '
            'alignment, embargo length, feature parity across splits.'
        )
else:
    print(f'[diagnostic] Trained through to planned iter {n_iters_planned}; '
          f'best_iter={best_iter}.')

print(
    f'Fit done in {fit_dt:.1f}s ({fit_dt/60:.1f} min); '
    f'val-best iter={best_iter}; tree_count={tree_count} '
    f'(use_best_model=True truncates to best_iter+1)'
)

model.save_model(str(MODEL_PATH))
_model_size_kb = MODEL_PATH.stat().st_size / 1024.0
print(f'Saved model: {MODEL_PATH}  ({_model_size_kb:,.1f} KB)')

Training CatBoost with params:
  loss_function: Logloss
  eval_metric: Logloss
  custom_metric: ['Logloss', 'AUC', 'PRAUC']
  iterations: 2000
  early_stopping_rounds: 100
  use_best_model: True
  auto_class_weights: None
  random_seed: 42
  verbose: 20
  thread_count: -1
  allow_writing_files: False
  boosting_type: Ordered
  has_time: True
  feature_border_type: GreedyLogSum
  border_count: 700
  grow_policy: SymmetricTree
  bootstrap_type: MVS
  leaf_estimation_method: Newton
  leaf_estimation_iterations: 10
  langevin: True
  learning_rate: 0.01
  l2_leaf_reg: 0.1
  depth: 6
  rsm: 1.0
  subsample: 1.0
  mvs_reg: 3.0
  diffusion_temperature: 10000
  random_strength: 0.0


[integrity] Pool row counts match: train=353,794, val=74,613
Fit start: 21:34:47


0:	learn: 0.6798266	test: 0.6797415	best: 0.6797415 (0)	total: 4.01s	remaining: 2h 13m 32s


20:	learn: 0.4844547	test: 0.4835338	best: 0.4835338 (20)	total: 1m 24s	remaining: 2h 12m 4s


40:	learn: 0.3803989	test: 0.3782654	best: 0.3782654 (40)	total: 2m 41s	remaining: 2h 8m 56s


...

500:	learn: 0.2210527	test: 0.2199120	best: 0.2191551 (463)	total: 32m 24s	remaining: 1h 36m 58s


520:	learn: 0.2207560	test: 0.2201362	best: 0.2191551 (463)	total: 33m 42s	remaining: 1h 35m 41s


540:	learn: 0.2212384	test: 0.2211337	best: 0.2191551 (463)	total: 35m	remaining: 1h 34m 26s


560:	learn: 0.2208243	test: 0.2207359	best: 0.2191551 (463)	total: 36m 19s	remaining: 1h 33m 10s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2191551335
bestIteration = 463

Shrink model to first 464 iterations.
[diagnostic] Early stopping fired at iter 463 (planned 2000, early_stopping_rounds=100).
[diagnostic] best_iter is < 30% of planned iterations — investigate data prep upstream: label/weight construction, train/val split alignment, embargo length, feature parity across splits.
Fit done in 2195.8s (36.6 min); val-best iter=463; tree_count=464 (use_best_model=True truncates to best_iter+1)
Saved model: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\catboost_model_1min.cbm  (835.3 KB)


## 5. Score val + test + train slice

In [6]:
# With use_best_model=True the saved model has been truncated to
# best_iter+1 trees, so plain predict_proba uses exactly the val-best
# ensemble — no ntree_end gymnastics needed. The Langevin/SGLB chain is
# still rich enough for the K=10 virtual-ensemble snapshots in
# Section 8 because best_iter is large under the legacy slow-learning
# configuration.
INFERENCE_NTREES = tree_count
print(f'Scoring with all {INFERENCE_NTREES} kept trees '
      f'(model truncated to val-best iter={best_iter} via use_best_model=True).')

print('Computing predictions on val + test ...')
p_val = model.predict_proba(val_df[feature_list].to_numpy())[:, 1]
p_test = model.predict_proba(test_df[feature_list].to_numpy())[:, 1]
print(f'  val p: min={p_val.min():.3f} med={np.median(p_val):.3f} max={p_val.max():.3f}')
print(f'  test p: min={p_test.min():.3f} med={np.median(p_test):.3f} max={p_test.max():.3f}')

print('Scoring training slice for train-period quantile table ...')
t0 = time.perf_counter()
p_train = model.predict_proba(train_df[feature_list].to_numpy())[:, 1]
print(
    f'  train p: min={p_train.min():.3f} med={np.median(p_train):.3f} '
    f'max={p_train.max():.3f}  scored in {time.perf_counter() - t0:.1f}s'
)
train_p_quantiles = {
    f'{q:.2f}': float(np.quantile(p_train, q)) for q in TRAIN_P_QUANTILE_LEVELS
}
train_p_max = float(p_train.max())

REGIME_COL = 'vol__rs__f__w240'
if REGIME_COL not in df.columns:
    raise RuntimeError(
        f'Regime feature {REGIME_COL!r} is not in the dataset columns. '
        "The cache contract requires this column to populate 'regime'. "
        'If the feature was renamed/removed, update REGIME_COL above '
        'to a current column name.'
    )

cache_frames = []
for name, sub, p in [('val', val_df, p_val), ('test', test_df, p_test)]:
    cache = pd.DataFrame({
        'k': sub['k'].to_numpy(),
        'ts': sub['ts'].to_numpy(),
        'y': sub['y'].astype(int).to_numpy(),
        'm_k': sub['m_k'].to_numpy(),
        'tau_k': sub['tau_k'].to_numpy() if 'tau_k' in sub.columns else np.full(len(sub), np.nan),
        'phi': sub['phi'].to_numpy() if 'phi' in sub.columns else np.full(len(sub), PHI),
        'regime': sub[REGIME_COL].to_numpy(dtype=float),
        'p': p.astype(float),
        'split': name,
    })
    cache_frames.append(cache)
cache_all = pd.concat(cache_frames, ignore_index=True)
cache_all.to_parquet(PREDICTIONS_PATH, index=False)
print(f'Saved predictions: {PREDICTIONS_PATH} ({len(cache_all):,} rows)')

Scoring with all 464 kept trees (model truncated to val-best iter=463 via use_best_model=True).
Computing predictions on val + test ...


  val p: min=0.007 med=0.060 max=0.984
  test p: min=0.012 med=0.091 max=0.940
Scoring training slice for train-period quantile table ...


  train p: min=0.004 med=0.059 max=0.980  scored in 2.5s


Saved predictions: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\research_predictions_1min.parquet (149,227 rows)


## 6. Block-bootstrap headline metrics

IID bootstrap is dropped from the headline at 1-min cadence: adjacent
rows share `M-1` future bars, so labels are mechanically correlated
within a horizon. IID CIs would be too tight by roughly `sqrt(M)`.
Block bootstrap with `block_size=M` resamples whole horizons and yields
the honest CI.

In [7]:
# Block bootstrap is mandatory at 1-min cadence. We deliberately do NOT
# report the IID leg in headline metrics because IID resampling treats
# overlapping 1-min labels as independent and underestimates CI width by
# roughly sqrt(M) (adjacent rows share M-1 future bars).
print(f'Computing headline metrics with block bootstrap (block_size=M={M}) ...')
os.makedirs(METRICS_PATH.parent, exist_ok=True)


def _ci(metric_name, fn, y, p):
    """Block-bootstrap CI for a metric on overlapping 1-min labels."""
    res_block = bootstrap_metric(
        fn, y, p, B=500, stratify=False, seed=0, block_size=M
    )
    return {
        'point': res_block.point,
        'block_ci': [res_block.ci_low, res_block.ci_high],
        'block_width': res_block.ci_high - res_block.ci_low,
    }


def _metrics(name, sub, p):
    y = sub['y'].astype(int).to_numpy()
    return {
        'split': name,
        'n_samples': int(len(sub)),
        'base_rate': float(y.mean()),
        'roc_auc': _ci('roc_auc', lambda y_, p_: float(roc_auc_score(y_, p_)), y, p),
        'pr_auc':  _ci('pr_auc',  lambda y_, p_: float(average_precision_score(y_, p_)), y, p),
        'log_loss': _ci('log_loss', lambda y_, p_: float(log_loss(y_, p_, labels=[0, 1])), y, p),
        'brier':    _ci('brier',   lambda y_, p_: float(brier_score_loss(y_, p_)), y, p),
    }


val_metrics = _metrics('val', val_df, p_val)
test_metrics = _metrics('test', test_df, p_test)

# Persist a serializable copy of the full CB_PARAMS dict for provenance.
_cb_params_serializable = {
    k: (list(v) if isinstance(v, tuple) else v) for k, v in CB_PARAMS.items()
}

out = {
    'config': {
        'embargo_k': int(EMBARGO_K),
        'iterations': int(CB_PARAMS['iterations']),
        'depth': int(CB_PARAMS['depth']),
        'l2_leaf_reg': float(CB_PARAMS['l2_leaf_reg']),
        'border_count': int(CB_PARAMS['border_count']),
        'learning_rate': float(CB_PARAMS['learning_rate']),
        'rsm': float(CB_PARAMS['rsm']),
        'subsample': float(CB_PARAMS['subsample']),
        'mvs_reg': float(CB_PARAMS['mvs_reg']),
        'random_strength': float(CB_PARAMS['random_strength']),
        'diffusion_temperature': float(CB_PARAMS['diffusion_temperature']),
        'use_best_model': bool(CB_PARAMS['use_best_model']),
        'early_stopping_rounds': int(CB_PARAMS.get('early_stopping_rounds', 0)),
        'has_time': bool(CB_PARAMS['has_time']),
        'boosting_type': str(CB_PARAMS['boosting_type']),
        'bootstrap_type': str(CB_PARAMS['bootstrap_type']),
        'langevin': bool(CB_PARAMS.get('langevin', False)),
        'cb_params_full': _cb_params_serializable,
        'weights': 'asymmetric_barrier_distance * lopez_de_prado_avg_uniqueness',
        'best_iteration_by_val_logloss': best_iter,
        'tree_count_saved': tree_count,
        'fit_seconds': fit_dt,
        'block_bootstrap_size': int(M),
    },
    'val': val_metrics,
    'test': test_metrics,
}
with open(METRICS_PATH, 'w') as f:
    json.dump(out, f, indent=2, default=str)

print('Headline metrics (1-min cadence; block-bootstrap CI, block_size=M):')
for split_name, m in [('val', val_metrics), ('test', test_metrics)]:
    print(f"  [{split_name}] n={m['n_samples']:,}  base_rate={m['base_rate']:.4f}")
    for metric_name in ['roc_auc', 'pr_auc', 'log_loss', 'brier']:
        mm = m[metric_name]
        print(
            f"      {metric_name:>9}: {mm['point']:.4f}  "
            f"block CI=[{mm['block_ci'][0]:.4f}, {mm['block_ci'][1]:.4f}]"
        )
print(f'\nSaved metrics: {METRICS_PATH}')

Computing headline metrics with block bootstrap (block_size=M=20) ...


Headline metrics (1-min cadence; block-bootstrap CI, block_size=M):
  [val] n=74,613  base_rate=0.1900
        roc_auc: 0.7621  block CI=[0.7504, 0.7745]
         pr_auc: 0.4302  block CI=[0.4066, 0.4562]
       log_loss: 0.4673  block CI=[0.4468, 0.4873]
          brier: 0.1447  block CI=[0.1379, 0.1516]
  [test] n=74,614  base_rate=0.2521
        roc_auc: 0.7605  block CI=[0.7469, 0.7720]
         pr_auc: 0.4832  block CI=[0.4628, 0.5025]
       log_loss: 0.5454  block CI=[0.5275, 0.5647]
          brier: 0.1792  block CI=[0.1729, 0.1861]

Saved metrics: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\analytics\research_metrics_1min.json


## 7. Persist predictions metadata

In [8]:
# Persisted here so every downstream notebook can pick an operating
# threshold from the training-period score distribution instead of
# peeking at val+test.
predictions_meta = {
    'label_cadence': '1min',
    'M': int(M),
    'PHI': float(PHI),
    'embargo_k': int(EMBARGO_K),
    'best_iteration_by_val_logloss': best_iter,
    'tree_count': tree_count,
    'fit_seconds': fit_dt,
    'n_train': int(len(train_df)),
    'n_val': int(len(val_df)),
    'n_test': int(len(test_df)),
    'train_p_min': float(p_train.min()),
    'train_p_median': float(np.median(p_train)),
    'train_p_max': train_p_max,
    'train_p_quantiles': train_p_quantiles,
    'cb_params': {
        'iterations': int(CB_PARAMS['iterations']),
        'depth': int(CB_PARAMS['depth']),
        'l2_leaf_reg': float(CB_PARAMS['l2_leaf_reg']),
        'border_count': int(CB_PARAMS['border_count']),
        'learning_rate': float(CB_PARAMS['learning_rate']),
        'rsm': float(CB_PARAMS['rsm']),
        'subsample': float(CB_PARAMS['subsample']),
        'mvs_reg': float(CB_PARAMS['mvs_reg']),
        'random_strength': float(CB_PARAMS['random_strength']),
        'diffusion_temperature': float(CB_PARAMS['diffusion_temperature']),
        'use_best_model': bool(CB_PARAMS['use_best_model']),
        'early_stopping_rounds': int(CB_PARAMS.get('early_stopping_rounds', 0)),
        'has_time': bool(CB_PARAMS['has_time']),
        'boosting_type': str(CB_PARAMS['boosting_type']),
        'bootstrap_type': str(CB_PARAMS['bootstrap_type']),
        'langevin': bool(CB_PARAMS.get('langevin', False)),
    },
    'weights': {
        'scheme': 'asymmetric_barrier_distance * lopez_de_prado_avg_uniqueness',
        'barrier_scheme': 'asymmetric_barrier_distance',
        'uniqueness_scheme': 'lopez_de_prado_avg_uniqueness',
        'uniqueness_normalize': False,
        'uniqueness_bar_stride': 1,
        'train_min': float(train_weights.min()),
        'train_max': float(train_weights.max()),
        'train_mean': float(train_weights.mean()),
        'train_sum': float(train_weights.sum()),
        'val_min': float(val_weights.min()),
        'val_max': float(val_weights.max()),
        'val_mean': float(val_weights.mean()),
        'val_sum': float(val_weights.sum()),
        'u_train_mean': float(u_train.mean()),
        'u_train_sum_effective_n': float(u_train.sum()),
        'u_val_mean': float(u_val.mean()),
        'u_val_sum_effective_n': float(u_val.sum()),
        'w_dist_train_mean': float(w_dist_train.mean()),
        'w_dist_val_mean': float(w_dist_val.mean()),
    },
    'git_sha': get_git_sha(),
}
with open(PREDICTIONS_META_PATH, 'w') as f:
    json.dump(predictions_meta, f, indent=2, default=str)
print(f'Saved predictions metadata: {PREDICTIONS_META_PATH}')
print(f'  train_p_quantiles: {train_p_quantiles}')

Saved predictions metadata: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\predictions_metadata_1min.json
  train_p_quantiles: {'0.05': 0.01567892112897951, '0.10': 0.0170963240894771, '0.50': 0.05941060113500241, '0.70': 0.10157915621890859, '0.75': 0.12210844797192538, '0.80': 0.15198725201728144, '0.85': 0.19375263512851573, '0.90': 0.2523930671728099, '0.95': 0.3537582125806059}


## 8. Virtual-ensemble uncertainty (train + val + test)

Save per-row epistemic uncertainty for every split so `04_offline_evaluation`
and `05_strategy_calibration` can read it from disk without re-running
CatBoost. CatBoost's virtual ensemble draws `VE_K` snapshots from the SGLB
posterior and we decompose the K-predictions into mean + epistemic
("knowledge") + aleatoric components.

Under the legacy stack (Langevin SGLB + Ordered boosting + Newton leaf
estimation + slow learning rate), `best_iter` is large and the chain
spans hundreds of stochastic trees, so VE diversity is healthy. If the
epistemic distribution collapses to ~0, suspect a configuration drift —
verify that `langevin=True`, `diffusion_temperature` is the legacy
10000, and the model was not truncated to a tiny `best_iter`.

In [9]:
# Train-slice VE: epistemic uncertainty + mean snapshot probability per row.
os.makedirs(TRAIN_UNC_PATH.parent, exist_ok=True)
print(f'Running virtual ensemble on training slice (K={VE_K}) ...')
t0 = time.perf_counter()
p_ve_train = virtual_ensemble_predictions(
    model,
    train_df[feature_list],
    virtual_ensembles_count=VE_K,
    feature_list=feature_list,
)
dec_train = predictive_uncertainty(p_ve_train)
dt = time.perf_counter() - t0
print(
    f'  VE done in {dt:.1f}s; mean_p_ve range '
    f'[{dec_train["mean_p"].min():.4f}, {dec_train["mean_p"].max():.4f}]'
)
print(
    f'  knowledge_unc range '
    f'[{dec_train["knowledge_uncertainty"].min():.6f}, '
    f'{dec_train["knowledge_uncertainty"].max():.6f}]'
)
print(
    f'  knowledge_unc median: '
    f'{np.median(dec_train["knowledge_uncertainty"]):.6f}'
)

train_unc_out = pd.DataFrame({
    'k': train_df['k'].to_numpy(),
    'ts': train_df['ts'].to_numpy(),
    'y': train_df['y'].astype(int).to_numpy(),
    'p_train': p_train.astype(float),
    'mean_p_ve_train': dec_train['mean_p'].astype(float),
    'knowledge_unc_train': dec_train['knowledge_uncertainty'].astype(float),
})
train_unc_out.to_parquet(TRAIN_UNC_PATH, index=False)
print(f'Saved: {TRAIN_UNC_PATH}  ({len(train_unc_out):,} rows)')

Running virtual ensemble on training slice (K=10) ...


  VE done in 4.6s; mean_p_ve range [0.0000, 1.0000]
  knowledge_unc range [0.000000, 0.693147]
  knowledge_unc median: 0.000000
Saved: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\analytics\train_scores_unc_1min.parquet  (353,794 rows)


In [10]:
# Val + test VE: same K=10 snapshots, written as a single parquet so
# 05_strategy_calibration can augment the predictions cache in one read.
print(f'Running virtual ensemble on val + test for cache augmentation ...')
val_test_rows = []
for name, sub in [('val', val_df), ('test', test_df)]:
    t0 = time.perf_counter()
    p_ve = virtual_ensemble_predictions(
        model,
        sub[feature_list],
        virtual_ensembles_count=VE_K,
        feature_list=feature_list,
    )
    dec = predictive_uncertainty(p_ve)
    dt = time.perf_counter() - t0
    print(f'  {name}: VE done in {dt:.1f}s; n={len(sub):,}')
    val_test_rows.append(pd.DataFrame({
        'k': sub['k'].to_numpy(),
        'split': name,
        'mean_p_ve': dec['mean_p'].astype(float),
        'knowledge_unc': dec['knowledge_uncertainty'].astype(float),
    }))
val_test_unc_out = pd.concat(val_test_rows, ignore_index=True)
val_test_unc_out.to_parquet(VAL_TEST_UNC_PATH, index=False)
print(f'Saved: {VAL_TEST_UNC_PATH}  ({len(val_test_unc_out):,} rows)')

# Quick sanity print: how many val+test rows pass the score gate at P_85,
# and what's the conditional knowledge_unc median there? Useful to confirm
# the epistemic gate has signal (median > 0 means it can discriminate;
# median == 0 means snapshots are too clustered).
p85 = float(np.quantile(p_train, 0.85))
mask_train_top = p_train >= p85
if mask_train_top.any():
    u_cond = dec_train['knowledge_uncertainty'][mask_train_top]
    print(
        f"  Among train p >= P_85={p85:.4f}: n={int(mask_train_top.sum()):,}, "
        f"knowledge_unc median = {float(np.median(u_cond)):.6f}, "
        f"p25 = {float(np.quantile(u_cond, 0.25)):.6f}"
    )

Running virtual ensemble on val + test for cache augmentation ...


  val: VE done in 0.6s; n=74,613


  test: VE done in 0.6s; n=74,614
Saved: C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\analytics\val_test_ve_unc_1min.parquet  (149,227 rows)
  Among train p >= P_85=0.1938: n=53,069, knowledge_unc median = 0.000000, p25 = 0.000000


## 9. Summary

In [11]:
print('=== Run summary ===')
print(f'  rows:  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'  embargo_k={EMBARGO_K}  M={M}  PHI={PHI}')
print(f'  best_iteration={best_iter}  fit_seconds={fit_dt:.1f}')
print()
print('Headline metrics (block bootstrap, block_size=M):')
for split_name, m in [('val', val_metrics), ('test', test_metrics)]:
    print(f"  [{split_name}] n={m['n_samples']:,}  base_rate={m['base_rate']:.4f}")
    for metric_name in ['roc_auc', 'pr_auc', 'log_loss', 'brier']:
        mm = m[metric_name]
        print(
            f"      {metric_name:>9}: {mm['point']:.4f}  "
            f"block CI=[{mm['block_ci'][0]:.4f}, {mm['block_ci'][1]:.4f}]"
        )
print()
print('Artifacts:')
print(f'  model:           {MODEL_PATH}')
print(f'  predictions:     {PREDICTIONS_PATH}')
print(f'  metrics:         {METRICS_PATH}')
print(f'  pred. metadata:  {PREDICTIONS_META_PATH}')
print(f'  train_p_max:     {train_p_max:.4f}')
print(f'  train_p_quantiles: {train_p_quantiles}')

=== Run summary ===
  rows:  train=353,794  val=74,613  test=74,614
  embargo_k=1200  M=20  PHI=0.0025
  best_iteration=463  fit_seconds=2195.8

Headline metrics (block bootstrap, block_size=M):
  [val] n=74,613  base_rate=0.1900
        roc_auc: 0.7621  block CI=[0.7504, 0.7745]
         pr_auc: 0.4302  block CI=[0.4066, 0.4562]
       log_loss: 0.4673  block CI=[0.4468, 0.4873]
          brier: 0.1447  block CI=[0.1379, 0.1516]
  [test] n=74,614  base_rate=0.2521
        roc_auc: 0.7605  block CI=[0.7469, 0.7720]
         pr_auc: 0.4832  block CI=[0.4628, 0.5025]
       log_loss: 0.5454  block CI=[0.5275, 0.5647]
          brier: 0.1792  block CI=[0.1729, 0.1861]

Artifacts:
  model:           C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\catboost_model_1min.cbm
  predictions:     C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\model_dataset\research_predictions_1min.parquet
  metrics:         C:\Users\vitil\OneDrive\Desktop\barrier_classifier\data\mod